## 实验2：CPU + CANN 异构环境验证体验

### 一、实验说明

#### 1.1 实验背景

随着人工智能技术向边缘端延伸，**异构计算**（Heterogeneous Computing）成为支撑各类AI应用的核心架构。异构计算是指在一个系统中集成多种类型的处理器（如：CPU、GPU、NPU等），通过任务分工与并行协作，充分发挥不同处理器的算力优势，从而在功耗、性能和成本之间取得最优平衡。

**华为CANN（Compute Architecture for Neural Networks）** 正是为实现这一目标而设计的异构计算架构。它向上对接PyTorch、TensorFlow、MindSpore等主流AI框架，向下管理昇腾AI处理器（NPU）、AI CPU、DVPP（数字视觉预处理）等异构计算资源，并提供AscendCL（昇腾计算语言）、ATC（昇腾张量编译器）等核心工具，形成承上启下的全栈软件体系。

**香橙派AIpro（8T）开发板** 是香橙派联合华为昇腾推出的高性能AI开发板，搭载4核64位ARM处理器和华为昇腾310系列AI处理器（NPU），具备8 TOPS INT8算力，是学习和实践CPU+NPU异构计算环境的理想平台。本实验将以香橙派AIpro开发板为载体，带领学生完整体验从CANN环境部署、ATC模型转换到ACL推理应用开发的全流程，建立对异构计算环境的系统认知和实践能力。

### 二、实验任务

#### 2.1 任务描述

本实验围绕“**CPU + NPU 异构环境验证**”这一核心目标，设计了三项递进式任务：

- **任务一：CANN异构计算环境部署**——在香橙派AIpro开发板上完成CANN Toolkit的安装与环境配置，验证NPU设备的可用性。
- **任务二：ATC模型转换**——使用昇腾张量编译器（ATC）将ONNX格式的深度学习模型转换为昇腾AI处理器支持的 `.om` 离线模型，理解模型转换的原理与流程。
- **任务三：ACL推理应用开发（含设备切换对比实验）**——基于AscendCL（pyACL）编写Python推理程序，加载 `.om` 模型完成图像分类推理，并设计设备切换对比实验：分别通过 `--device` 配置将任务分流至**CPU**和**NPU**执行，使用 `time` 命令记录推理耗时，直观对比两种处理器的性能差异。

通过完成上述任务，学生将亲身实践“部署 → 转换 → 推理”的完整开发链路，掌握在异构计算平台上构建AI应用的核心能力。


#### 2.2 学习目标

知识与理解：
- 理解异构计算的基本概念以及CANN在昇腾AI生态中的定位与核心组件作用。
- 掌握ATC模型转换工具的工作原理及 `.om` 模型的生成流程。
- 了解AscendCL（pyACL）编程模型的基本要素（Device、Context、Stream）。
- 理解工具链协同关系：ATC负责将开源框架模型转换为NPU专用格式，ACL负责运行时推理调度，两者共同构成从模型优化到推理执行的完整链路。

技能与实践：
- 能够在香橙派AIpro上独立完成CANN环境的部署与验证（**CANNLab在线实验忽略**）。
- 能够使用ATC工具完成ONNX模型到 `.om` 模型的转换。
- 能够基于pyACL API编写具备资源管理、内存拷贝和推理执行的Python程序。
- 能够通过设备切换实验验证NPU与CPU的推理性能差异，掌握性能对比实验方法。

### 三、任务准备

#### 3.1 前置知识

- **Python编程基础**：变量、函数、NumPy数组操作、OpenCV图像处理基础。
- **深度学习基础**：了解卷积神经网络（CNN）的基本结构，熟悉图像分类任务流程。
- **Linux命令行操作**：掌握 `cd`、`ls`、`su`、`wget`、`chmod`、`source` 等常用命令。
- **SSH远程连接**：会使用终端工具连接远程设备。
- **昇腾生态概念**：
  - **CANN**：昇腾AI处理器的异构计算架构，负责模型编译、算子执行与异构调度。
  - **ATC**：昇腾张量编译器，将开源框架模型转换为 `.om` 离线模型。
  - **AscendCL（ACL）**：昇腾计算语言，提供C++/Python API，管理Device、Context、Stream、内存及模型推理。
  - **pyACL**：AscendCL的Python封装，便于快速编写推理程序。

#### 3.2 实验数据准备

**1. 获取测试图片**
在开发板终端执行以下命令下载测试用图片：
```bash
mkdir -p $HOME/experiment/lab_02
cd $HOME/experiment/lab_02
wget https://obs-9be7.obs.cn-east-2.myhuaweicloud.com/models/aclsample/dog1_1024_683.jpg
wget https://obs-9be7.obs.cn-east-2.myhuaweicloud.com/models/aclsample/dog2_1024_683.jpg
```


**2. 获取ONNX模型**
```bash
wget https://obs-9be7.obs.cn-east-2.myhuaweicloud.com/003_Atc_Models/resnet50/resnet50.onnx
```

### 四、任务实施

#### 4.1 任务一：CANN异构计算环境部署（**CANNLab在线实验环境下跳过这一任务**）

> **注意**：香橙派AIpro出厂镜像通常预装了CANN商业版。本步骤演示如何升级到最新社区版。如果出厂镜像已满足需求，可跳至步骤4验证环境。

**步骤1：切换至root用户并清理旧版本**
```bash
su
# 输入root密码（默认为root）
cd /usr/local/Ascend/ascend-toolkit/
rm -rf *    # 删除旧版本释放磁盘空间
```

**步骤2：下载并安装最新版CANN Toolkit**
从昇腾社区资源下载中心获取最新版本，或直接在终端执行：
```bash
cd /home/HwHiAiUser/Downloads
# 下载CANN Toolkit安装包（以下为示例URL，请替换为实际版本）
wget https://ascend-repo.obs.cn-east-2.myhuaweicloud.com/CANN/...
chmod +x ./Ascend-cann-toolkit-*.run
./Ascend-cann-toolkit-*.run --install
```
安装过程中有交互提示时输入 `Y` 确认。

**步骤3：配置环境变量**
```bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh
# 建议将环境变量永久写入配置文件
echo "source /usr/local/Ascend/ascend-toolkit/set_env.sh" >> ~/.bashrc
```

**步骤4：验证安装**
```bash
npu-smi info
```
正常输出应显示NPU芯片名称、温度、内存使用情况和固件版本等信息。若显示 `npu-smi: command not found`，请检查环境变量配置是否正确。

```bash
atc --help
```
正常输出应显示ATC工具的帮助信息及参数说明。

#### 4.2 任务二：ATC模型转换

**步骤1：确认模型文件**
```bash
ls $HOME/experiment/lab_02/resnet50.onnx
```
确认ONNX模型已准备完毕。

**步骤2：查询NPU芯片版本**
```bash
npu-smi info
```
记录输出中的芯片名称信息（如 `310B`），用于 `--soc_version` 参数。


**步骤3：执行ATC模型转换**
```bash
cd $HOME/experiment/lab_02
atc --model=resnet50.onnx \
    --framework=5 \
    --output=resnet50 \
    --input_shape="actual_input_1:1,3,224,224" \
    --soc_version=Ascend310B
```

**关键参数说明：**

| 参数 | 含义 |
|------|------|
| `--model` | 待转换的ONNX模型路径 |
| `--framework` | 原始框架类型：0=Caffe, 3=TensorFlow, 5=ONNX |
| `--output` | 输出 `.om` 模型的文件名（无需加后缀）|
| `--input_shape` | 模型输入数据的维度 |
| `--soc_version` | 昇腾AI处理器版本（根据 `npu-smi info` 填写）|

**步骤4：验证转换结果**
转换成功后终端显示 `ATC run success`，并检查：
```bash
ls -lh $HOME/experiment/lab_02/resnet50.om
```
确认 `.om` 文件已生成。

**常见问题处理**：
- **ATC命令报错 `command not found`**：确认已执行 `source /usr/local/Ascend/ascend-toolkit/set_env.sh`，使环境变量生效。
- **转换过程卡住**：内存不足是常见原因，可创建swap分区解决。执行：
  ```bash
  fallocate -l 8G /swapfile
  chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile
  ```

#### 4.3 任务三：ACL推理应用开发（含设备切换对比实验）

本任务要求学生编写完整的Python推理程序，实现资源初始化、模型加载、数据预处理、推理执行和结果解析全流程。


**步骤1：编写ACL推理程序**
创建 `resnet50_inference.py` 文件，内容如下：

```python
import os
import sys
import acl
import numpy as np
from PIL import Image
import time

# --------------------- 常量定义 ---------------------
ACL_MEM_MALLOC_HUGE_FIRST = 0
ACL_MEMCPY_HOST_TO_DEVICE = 1
ACL_MEMCPY_DEVICE_TO_HOST = 2
MODEL_PATH = os.path.expanduser("/home/developer/experiment/lab_02/resnet50.om")
IMG_PATH = os.path.expanduser("/home/developer/experiment/lab_02/dog1_1024_683.jpg")

def preprocess(image_path):
    """预处理图像：缩放、裁剪、归一化、通道转换"""
    image = Image.open(image_path).convert('RGB')
    image = image.resize((256, 256))
    image = image.crop((16, 16, 240, 240))  # 中心裁剪到224x224
    image = np.array(image, dtype=np.float32)
    # 归一化：均值[0.485, 0.456, 0.406] 标准差[0.229, 0.224, 0.225]
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    image = (image / 255.0 - mean) / std
    image = image.transpose(2, 0, 1)        # HWC → CHW
    image = np.expand_dims(image, axis=0)   # CHW → NCHW
    return np.ascontiguousarray(image)

def main():
    # 1. ACL初始化
    ret = acl.init()
    assert ret == 0, "ACL初始化失败"

    # 2. 打开设备并创建Context
    device_id = 0
    ret = acl.rt.set_device(device_id)
    context, ret = acl.rt.create_context(device_id)
    assert ret == 0, "创建Context失败"

    # 3. 加载模型
    model_id, ret = acl.mdl.load_from_file(MODEL_PATH)
    model_desc = acl.mdl.create_desc()
    ret = acl.mdl.get_desc(model_desc, model_id)

    # 4. 创建输入数据集
    input_data = preprocess(IMG_PATH)
    input_size = input_data.nbytes
    input_buffer, ret = acl.rt.malloc(input_size, ACL_MEM_MALLOC_HUGE_FIRST)
    src_ptr = acl.util.bytes_to_ptr(input_data.tobytes())
    ret = acl.rt.memcpy(input_buffer, input_size,
                        src_ptr, input_size,
                        ACL_MEMCPY_HOST_TO_DEVICE)

    input_dataset = acl.mdl.create_dataset()
    input_data_buffer = acl.create_data_buffer(input_buffer, input_size)
    _, ret = acl.mdl.add_dataset_buffer(input_dataset, input_data_buffer)

    # 5. 准备输出数据集
    output_size = acl.mdl.get_output_size_by_index(model_desc, 0)
    output_buffer, ret = acl.rt.malloc(output_size, ACL_MEM_MALLOC_HUGE_FIRST)
    output_dataset = acl.mdl.create_dataset()
    output_data_buffer = acl.create_data_buffer(output_buffer, output_size)
    _, ret = acl.mdl.add_dataset_buffer(output_dataset, output_data_buffer)

    # 6. 执行推理并计时
    start = time.time()
    ret = acl.mdl.execute(model_id, input_dataset, output_dataset)
    end = time.time()

    if ret == 0:
        # 7. 获取输出结果
        result_buffer, ret = acl.rt.malloc_host(output_size)
        ret = acl.rt.memcpy(result_buffer, output_size,
                            output_buffer, output_size,
                            ACL_MEMCPY_DEVICE_TO_HOST)

        result_bytes = acl.util.ptr_to_bytes(result_buffer, output_size)
        result = np.frombuffer(result_bytes, dtype=np.byte)

        # 输出Top-5分类ID
        top5_idx = np.argsort(result)[-5:][::-1]
        print(f"推理完成！耗时: {(end - start) * 1000:.2f} ms")
        print(f"Top-5 分类ID: {top5_idx}")
    else:
        print(f"推理执行失败，错误码: {ret}")

    # 8. 释放资源
    acl.mdl.destroy_dataset(output_dataset)
    acl.mdl.destroy_dataset(input_dataset)
    acl.rt.free(output_buffer)
    acl.rt.free(input_buffer)
    acl.destroy_data_buffer(input_data_buffer)
    acl.destroy_data_buffer(output_data_buffer)
    acl.mdl.destroy_desc(model_desc)
    acl.mdl.unload(model_id)
    acl.rt.destroy_context(context)
    acl.rt.reset_device(device_id)
    acl.finalize()

if __name__ == "__main__":
    main()
```

**步骤2：运行程序**
```bash
cd $HOME/experiment/lab_02
python3 resnet50_inference.py
```


In [3]:
import os
import sys
import acl
import numpy as np
from PIL import Image
import time

# --------------------- 常量定义 ---------------------
ACL_MEM_MALLOC_HUGE_FIRST = 0
ACL_MEMCPY_HOST_TO_DEVICE = 1
ACL_MEMCPY_DEVICE_TO_HOST = 2
MODEL_PATH = os.path.expanduser("/home/developer/experiment/lab_02/resnet50.om")
IMG_PATH = os.path.expanduser("/home/developer/experiment/lab_02/dog1_1024_683.jpg")

def preprocess(image_path):
    """预处理图像：缩放、裁剪、归一化、通道转换"""
    image = Image.open(image_path).convert('RGB')
    image = image.resize((256, 256))
    image = image.crop((16, 16, 240, 240))  # 中心裁剪到224x224
    image = np.array(image, dtype=np.float32)
    # 归一化：均值[0.485, 0.456, 0.406] 标准差[0.229, 0.224, 0.225]
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    image = (image / 255.0 - mean) / std
    image = image.transpose(2, 0, 1)        # HWC → CHW
    image = np.expand_dims(image, axis=0)   # CHW → NCHW
    return np.ascontiguousarray(image)

def main():
    # 1. ACL初始化
    ret = acl.init()
    
    # 2. 打开设备并创建Context
    device_id = 0
    ret = acl.rt.set_device(device_id)
    context, ret = acl.rt.create_context(device_id)
    
    # 3. 加载模型
    model_id, ret = acl.mdl.load_from_file(MODEL_PATH)
    model_desc = acl.mdl.create_desc()
    ret = acl.mdl.get_desc(model_desc, model_id)

    # 4. 创建输入数据集
    input_data = preprocess(IMG_PATH)
    input_size = input_data.nbytes
    input_buffer, ret = acl.rt.malloc(input_size, ACL_MEM_MALLOC_HUGE_FIRST)
    src_ptr = acl.util.bytes_to_ptr(input_data.tobytes())
    ret = acl.rt.memcpy(input_buffer, input_size,
                        src_ptr, input_size,
                        ACL_MEMCPY_HOST_TO_DEVICE)

    input_dataset = acl.mdl.create_dataset()
    input_data_buffer = acl.create_data_buffer(input_buffer, input_size)
    _, ret = acl.mdl.add_dataset_buffer(input_dataset, input_data_buffer)

    # 5. 准备输出数据集
    output_size = acl.mdl.get_output_size_by_index(model_desc, 0)
    output_buffer, ret = acl.rt.malloc(output_size, ACL_MEM_MALLOC_HUGE_FIRST)
    output_dataset = acl.mdl.create_dataset()
    output_data_buffer = acl.create_data_buffer(output_buffer, output_size)
    _, ret = acl.mdl.add_dataset_buffer(output_dataset, output_data_buffer)

    # 6. 执行推理并计时
    start = time.time()
    ret = acl.mdl.execute(model_id, input_dataset, output_dataset)
    end = time.time()

    # 7. 获取输出结果
    result_buffer, ret = acl.rt.malloc_host(output_size)
    ret = acl.rt.memcpy(result_buffer, output_size,
                        output_buffer, output_size,
                        ACL_MEMCPY_DEVICE_TO_HOST)

    result_bytes = acl.util.ptr_to_bytes(result_buffer, output_size)
    result = np.frombuffer(result_bytes, dtype=np.byte)

    # 输出Top-5分类ID
    top5_idx = np.argsort(result)[-5:][::-1]
    print(f"推理完成！耗时: {(end - start) * 1000:.2f} ms")
    print(f"Top-5 分类ID: {top5_idx}")
    
    # 8. 释放资源
    acl.mdl.destroy_dataset(output_dataset)
    acl.mdl.destroy_dataset(input_dataset)
    acl.rt.free(output_buffer)
    acl.rt.free(input_buffer)
    acl.destroy_data_buffer(input_data_buffer)
    acl.destroy_data_buffer(output_data_buffer)
    acl.mdl.destroy_desc(model_desc)
    acl.mdl.unload(model_id)
    acl.rt.destroy_context(context)
    acl.rt.reset_device(device_id)
    acl.finalize()

In [4]:
main()

推理完成！耗时: 0.77 ms
Top-5 分类ID: [3999 2147 2107 2111 2115]


**步骤3：设计并执行设备切换对比实验**
修改上述代码中步骤2的设备设置部分（`acl.rt.set_device(device_id)` 部分），增加以下逻辑：

```python
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--device", type=str, default="npu",
                    choices=["npu", "cpu"],
                    help="选择推理设备：npu（昇腾AI处理器）或 cpu（ARM CPU）")
args = parser.parse_args()

if args.device == "npu":
    # 使用NPU推理（流程同上）
    ret = acl.rt.set_device(0)
    context, ret = acl.rt.create_context(0)
    # ... 加载 .om 模型，执行推理
else:
    # 使用CPU推理：此处加载原始ONNX模型，基于CPU后端执行
    import onnxruntime as ort
    session = ort.InferenceSession(MODEL_ONNX_PATH)
    input_name = session.get_inputs()[0].name

    if input_data.dtype == np.float64:
        input_data = input_data.astype(np.float32)

    if input_data.shape[0] != 16:
        input_data = np.repeat(input_data, 16, axis=0)

    start = time.time()
    output = session.run(None, {input_name: input_data})[0]
    end = time.time()

    top5_idx = np.argsort(output.flatten())[-5:][::-1]
    print(f"CPU推理耗时: {(end - start) * 1000:.2f} ms")
    print(f"Top-5 分类ID: {top5_idx}")

    return
```

**对比实验执行与记录：**

| 实验编号 | 设备类型 | CLI调用命令（示例） | 推理耗时 | Top-5分类ID | 备注 |
|----------|----------|-------------------|----------|-------------|------|
| 1 | NPU | `time python3 resnet50_inference.py --device npu` | 待记录 | 待记录 | 测量真实推理耗时 |
| 2 | CPU | `time python3 resnet50_inference.py --device cpu` | 待记录 | 待记录 | 对比分析 |

```bash
# 实验1：NPU推理
time python3 resnet50_inference.py --device npu

# 实验2：CPU推理
time python3 resnet50_inference.py --device cpu
```

In [10]:
MODEL_ONNX_PATH = os.path.expanduser("/home/developer/experiment/lab_02/resnet50.onnx")

def main():
    input_data = preprocess(IMG_PATH)
    input_data = np.repeat(input_data, 16, axis=0)
    input_size = input_data.nbytes

    device = "cpu"

    if device == "npu":
        ret = acl.init()
        
        device_id = 0
        ret = acl.rt.set_device(device_id)
        context, ret = acl.rt.create_context(device_id)
    else:
        import onnxruntime as ort

        session = ort.InferenceSession(MODEL_ONNX_PATH)
        input_name = session.get_inputs()[0].name
        
        if input_data.dtype == np.float64:
            input_data = input_data.astype(np.float32)

        if input_data.shape[0] != 16:
            input_data = np.repeat(input_data, 16, axis=0)
            
        start = time.time()
        output = session.run(None, {input_name: input_data})[0]
        end = time.time()

        top5_idx = np.argsort(output.flatten())[-5:][::-1]
        print(f"CPU inference time cost: {(end - start) * 1000:.2f} ms")
        print(f"Top-5 classification ID: {top5_idx}")

        return

    model_id, ret = acl.mdl.load_from_file(MODEL_PATH)
    model_desc = acl.mdl.create_desc()
    ret = acl.mdl.get_desc(model_desc, model_id)
    
    start = time.time()

    input_buffer, ret = acl.rt.malloc(input_size, ACL_MEM_MALLOC_HUGE_FIRST)
    
    src_ptr = acl.util.bytes_to_ptr(input_data.tobytes())
    ret = acl.rt.memcpy(input_buffer, input_size, src_ptr, input_size, ACL_MEMCPY_HOST_TO_DEVICE)
    
    input_dataset = acl.mdl.create_dataset()
    input_data_buffer = acl.create_data_buffer(input_buffer, input_size)
    _, ret = acl.mdl.add_dataset_buffer(input_dataset, input_data_buffer)
    
    output_size = acl.mdl.get_output_size_by_index(model_desc, 0)
    output_buffer, ret = acl.rt.malloc(output_size, ACL_MEM_MALLOC_HUGE_FIRST)
    
    output_dataset = acl.mdl.create_dataset()
    output_data_buffer = acl.create_data_buffer(output_buffer, output_size)
    _, ret = acl.mdl.add_dataset_buffer(output_dataset, output_data_buffer)
    
    ret = acl.mdl.execute(model_id, input_dataset, output_dataset)
    end = time.time()

    result_buffer, ret = acl.rt.malloc_host(output_size)
    ret = acl.rt.memcpy(result_buffer, output_size, output_buffer, output_size, ACL_MEMCPY_DEVICE_TO_HOST)
    result_bytes = acl.util.ptr_to_bytes(result_buffer, output_size)
    result = np.frombuffer(result_bytes, dtype=np.float32)

    top5_idx = np.argsort(result)[-5:][::-1]
    print(f"Inference finished! time cost: {(end - start) * 1000:.2f} ms")
    print(f"Top-5 classification ID: {top5_idx}")
    
    acl.mdl.destroy_dataset(output_dataset)
    acl.mdl.destroy_dataset(input_dataset)

    acl.rt.free(output_buffer)
    acl.rt.free(input_buffer)

    acl.destroy_data_buffer(input_data_buffer)
    acl.destroy_data_buffer(output_data_buffer)

    acl.mdl.destroy_desc(model_desc)
    acl.mdl.unload(model_id)

    acl.rt.destroy_context(context)
    acl.rt.reset_device(device_id)

    acl.finalize()

In [12]:
main()

CPU inference time cost: 788.78 ms
Top-5 classification ID: [ 5162  4162 14162  3162  9162]


#### 4.4 重要参数说明

| 参数/接口 | 含义 |
|-----------|------|
| `acl.init()` | 初始化ACL运行环境 |
| `acl.rt.set_device(0)` | 指定计算设备（NPU） |
| `acl.rt.malloc()` | 分配Device侧内存 |
| `acl.rt.memcpy()` | Host与Device间的数据拷贝 |
| `acl.mdl.load_from_file()` | 加载 `.om` 离线模型 |
| `acl.mdl.execute()` | 执行模型推理 |
| `--framework` | 原始框架类型（0=Caffe, 3=TensorFlow, 5=ONNX）|
| `--soc_version` | 昇腾AI处理器版本号 |

### 五、任务拓展

完成基础实验后，鼓励学生进行以下拓展探索：

**拓展1：多模型性能对比**
更换不同架构的分类模型（如MobileNetV3、EfficientNet-B0），使用ATC转换为 `.om` 模型后进行推理，记录各模型在NPU上的推理耗时与准确率。可从ONNX Model Zoo或PyTorch Hub获取预训练模型，导出为ONNX格式后按相同流程测试。

**拓展2：DVPP预处理加速**
在上述代码中仅使用了Python图像库进行预处理，学生可尝试使用CANN提供的DVPP（数字视觉预处理）硬件加速模块完成图像缩放和色彩空间转换，并对比与OpenCV/Pillow软件预处理的性能差异。DVPP可通过pyACL的 `acl.media` 模块调用。

**拓展3：多Stream并行推理**
在单一Stream单任务的基础上，创建多个Stream，将多张图像分别发送到不同Stream中并行推理，对比单Stream和多Stream的吞吐量差异。需在代码中创建多个Stream实例，通过 `acl.rt.create_stream()` 创建，使用 `acl.rt.launch_callback()` 或异步接口管理并发任务。

**拓展4：自定义模型部署**
鼓励学生基于自身研究方向，将自训练的PyTorch模型（如文本分类模型、目标检测模型）导出为ONNX格式，通过本实验完整流程部署到香橙派AIpro上，验证模型在边缘设备上的推理效果。

**拓展5：异构调度策略实验**
通过 `taskset` 命令将不同进程绑定到不同CPU核心，观察CANN Runtime如何在不同CPU核心和NPU之间调度计算任务，分析异构调度的效率与机制。


### 六、实验总结

本实验围绕“CPU + CANN 异构环境验证体验”这一主题，基于香橙派AIpro开发板，带领学生完成了从环境部署到推理应用开发的完整实践。

**实验核心收获：**
1. **异构计算认知**：通过CPU与NPU推理的性能对比实验，直观感受了异构计算“任务分工、优势互补”的设计理念——NPU擅长大规模矩阵运算，处理并行计算密集型任务；CPU则适合承担流程控制与通用计算任务。两种处理器的协同工作，是在边缘场景实现高效AI推理的关键。
2. **工具链理解**：通过实际操作ATC（模型转换）、ACL（推理调度）等核心工具，理解了CANN软件栈在昇腾AI生态中的枢纽作用。ATC将开源框架模型转换为NPU专用格式，ACL则负责运行时资源管理和推理执行，两者共同构成了高效的模型部署工具体系。
3. **部署能力**：掌握了CANN环境安装、环境变量配置、ATC模型转换、ACL推理程序编写和常见安装问题排查等一整套边缘AI应用部署方法，具备了在异构计算平台上构建AI应用的工程实践能力。

| 学习阶段 | 核心内容 | 掌握技能 |
|----------|----------|----------|
| 环境部署 | CANN Toolkit安装与验证 | 异构计算环境的搭建能力 |
| 模型转换 | ATC工具将ONNX模型转.om | 跨框架模型迁移与优化能力 |
| 推理开发 | pyACL API编写推理程序 | 昇腾平台AI推理应用开发能力 |
| 性能对比 | CPU vs NPU推理耗时测试 | 异构平台性能评估与分析能力 |

**拓展思考方向：**
- 在资源受限的边缘计算场景下，如何根据任务负载特点合理分配CPU与NPU的计算资源？
- 关键原则是“计算密集型任务交由NPU，控制流程与轻量级处理留在CPU”。例如在大规模图像识别、自然语言推理等场景，将模型推理完全卸载到NPU，CPU专注于接收数据、启停任务和返回结果，从而实现吞吐最大化。
- 若引入多Stream并发和DVPP预处理模块，推理效率还能获得多大程度的提升？

本次实验为学生后续深入学习模型量化、算子开发、分布式推理等高级主题奠定了坚实的基础。建议学生在理解本实验基础操作之上，利用昇腾社区的丰富资源（如CANN文档、昇腾社区样例仓）进一步拓展实践深度。